# 02 · LoRA Fine-Tuning of Turkish GPT-2

Instruction fine-tuning of **`ytu-ce-cosmos/turkish-gpt2-medium`** and **`turkish-gpt2-large`** with LoRA adapters (PEFT) and TRL's `SFTTrainer`, on two versions of the same 13,891 questions:

| Run | Base model | Training answers | Notebook cells |
|---|---|---|---|
| 1 | GPT-2 medium | GPT-4o | Run 1 |
| 2 | GPT-2 large | GPT-4o | Run 2 |
| 3 | GPT-2 medium | DeepSeek | Run 3 |
| 4 | GPT-2 large | DeepSeek | Run 4 |

**Setup:** LoRA `r=16`, `alpha=32`, `dropout=0.05`, target modules `c_attn` and `c_proj` · learning rate `2e-4` · 3 epochs · effective batch size 8 · bf16.

Prompt format (Turkish): `Soru: <question>\nCevap: <answer>` ("Question: … Answer: …").

> Requires a CUDA GPU. The JSONL training files are created by `01_dataset_analysis.ipynb`.

## Run 1 · GPT-2 medium on GPT-4o answers

In [29]:
# 1. Import libraries
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer
from peft import LoraConfig
from datasets import load_dataset
import torch

# 2. Load model and tokenizer (MEDIUM model)
model_id = "ytu-ce-cosmos/turkish-gpt2-medium"

tokenizer = AutoTokenizer.from_pretrained(model_id)
# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     device_map="auto",
#     torch_dtype=torch.float16,
#     load_in_4bit=True
# )
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,  # 4-bit loading removed
    load_in_8bit=False,  # 👈 ADDED
    load_in_4bit=False   # 👈 ADDED
)
# 3. Load the dataset (V1: GPT-4o dataset)
dataset_gpt4o = load_dataset("json", data_files="v1_dataset_gpt4o.jsonl", split="train")
# Copy the "response" column as "completion"
dataset_gpt4o = dataset_gpt4o.map(lambda example: {"completion": example["response"]})

In [30]:
# 4. Set up the LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["c_attn", "c_proj"],  # 👈 changed
    # lora_dropout=0.05,  # note: dropout was accidentally disabled in this first run; runs 2-4 use 0.05
    bias="none",
    task_type="CAUSAL_LM"
)
# 5. Training settings
training_args = TrainingArguments(
    output_dir="./gpt2-medium-finetuned-gpt4o",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    num_train_epochs=3,
    bf16=True,   # Use fp16 here if the GPU does not support bf16
)


# ⚡Formatting function
def formatting_prompts(example):
    return f"Soru: {example['prompt']}\nCevap: {example['response']}"

# 6. Trainer setup
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_gpt4o,
    peft_config=lora_config,
    formatting_func=formatting_prompts,  # 👈 NEW
    args=training_args,
)

# 7. Start training
trainer.train()

Step,Training Loss
10,2.766000
20,2.639900
30,2.568600
40,2.588800
50,2.576000
60,2.451200
70,2.546000
80,2.561700
90,2.514000
100,2.565600


TrainOutput(global_step=5208, training_loss=2.299475330727807, metrics={'train_runtime': 1749.9021, 'train_samples_per_second': 23.814, 'train_steps_per_second': 2.976, 'total_flos': 1.58636083120128e+16, 'train_loss': 2.299475330727807})

In [31]:
trainer.model.save_pretrained("./gpt2-medium-finetuned-gpt4o-final")
tokenizer.save_pretrained("./gpt2-medium-finetuned-gpt4o-final")

('./gpt2-medium-finetuned-gpt4o-final/tokenizer_config.json',
 './gpt2-medium-finetuned-gpt4o-final/special_tokens_map.json',
 './gpt2-medium-finetuned-gpt4o-final/vocab.json',
 './gpt2-medium-finetuned-gpt4o-final/merges.txt',
 './gpt2-medium-finetuned-gpt4o-final/added_tokens.json',
 './gpt2-medium-finetuned-gpt4o-final/tokenizer.json')

### Qualitative check with a single question

In [32]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 1. Load the model and tokenizer
model_path = "./gpt2-medium-finetuned-gpt4o-final"

model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_path)

# 2. Write a test question
input_text = "Soru: Osmanlı Devleti'nin kuruluş yılı nedir?\nCevap:"

# 3. Tokenize
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

# 4. Generate an answer from the model
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)

# 5. Decode the output
output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# 6. Show the result
print("\nModel answer:\n")
print(output_text)


Model answer:

Soru: Osmanlı Devleti'nin kuruluş yılı nedir?
Cevap: Osmanlı Devleti'nin kuruluş yılı, 13. yüzyılın sonları veya 14. yüzyılın başlarında, Osmanlı İmparatorluğu'nun Bizans İmparatorluğu ile savaşarak Osmanlı İmparatorluğu'nun temellerini attığı bir tarihtir. Bu tarih, Osmanlı İmparatorluğu'nun tarihinin başlangıcı olarak kabul edilir.

Osmanlı Devleti'nin kuruluşu, Osmanlı İmparatorluğu'nun siyasi, askeri ve kültürel gelişiminin önemli bir aşamasını oluşturur. Bu tarih, Osmanlı İmparatorluğu'nun yükselişinin ve modernleşme sürecinin önemli bir dönüm noktası olmuştur.

Osmanlı Devleti'nin kuruluşu, Osmanlı


In [33]:
input_text = "Soru: Osmanlı Devleti'nin kuruluş yılı nedir? Kısa ve net bir cevap ver.\nCevap:"

# Tokenize
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

# Generate answer from the model
outputs = model.generate(
    **inputs,
    max_new_tokens=50,  # Generate fewer tokens because we want short answers
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)

# Decode output
output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Print
print("\nModel answer:\n")
print(output_text)


Model answer:

Soru: Osmanlı Devleti'nin kuruluş yılı nedir? Kısa ve net bir cevap ver.
Cevap: Osmanlı Devleti'nin kuruluş tarihi kesin olarak bilinmemektedir. Ancak, bazı kaynaklar Osmanlı'nın daha erken bir tarihte Osmanlı Devleti'nin bir parçası haline geldiğini iddia etmektedir.

Osmanlı Devleti, 1361 yılında Yıldırım Bayezid tarafından Yıldırım Bayezid Han tarafından kurulmuştur. Bu


In [34]:
# Our test questions
sorular = [
    "Türkiye'nin en uzun nehri nedir?",
    "İstanbul'un fethi kaç yılında gerçekleşmiştir?",
    "DNA nedir?",
    "Güneş Sistemi'nde kaç gezegen vardır?",
    "Bir insanın normal vücut sıcaklığı kaç derecedir?",
    "Atatürk hangi yıl doğdu?"
]

# Ask the model the questions
for soru in sorular:
    input_text = f"Soru: {soru} Kısa ve net bir cevap ver.\nCevap:"
    
    # Tokenize
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    # Generate answer
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

    # Decode
    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Print result
    print("\n============================")
    print(f"Soru: {soru}")
    print("Model answer:")
    print(output_text)
    print("============================\n")


Soru: Türkiye'nin en uzun nehri nedir?
Model answer:
Soru: Türkiye'nin en uzun nehri nedir? Kısa ve net bir cevap ver.
Cevap: Fırat Nehri.

Fırat Nehri, Türkiye'nin en uzun nehridir. Türkiye'nin yaklaşık olarak 4.760 kilometre (2.440 mil) uzunluğundadır.

Fırat Nehri, Türkiye'nin en uzun nehridir, ancak


Soru: İstanbul'un fethi kaç yılında gerçekleşmiştir?
Model answer:
Soru: İstanbul'un fethi kaç yılında gerçekleşmiştir? Kısa ve net bir cevap ver.
Cevap: İstanbul'un fethi, 532 yılında gerçekleşmiştir. Bu tarih, Bizans'ın Konstantinopolis'i fethettiği tarihtir.

Fetih, Bizans'ın, İstanbul'u fethetmesi ve onu Roma İmparatorluğu'nun başkenti yapmasıdır.


Soru: DNA nedir?
Model answer:
Soru: DNA nedir? Kısa ve net bir cevap ver.
Cevap: DNA, proteinler gibi genetik materyaldir.

DNA, hücrelerde bulunan temel yapı taşıdır ve proteinlerin aksine, genetik bilgiyi taşıyan tek yapı taşıdır.

DNA, genetik bilgiyi taşıyan tek yapı taşıdır ve hücre içinde,


Soru: Güneş Sistemi'nde kaç gezegen 

### Test set: 50 factual questions across 8 categories
History, science, geography, general knowledge, health, literature, math and technology.

In [35]:
# TEST QUESTIONS (50)
sorular = [
    # 📚 History
    "İstanbul'un fethi hangi yılda gerçekleşmiştir?",
    "Atatürk hangi yılda doğmuştur?",
    "Magna Carta hangi yıl imzalanmıştır?",
    "Fransız Devrimi kaç yılında olmuştur?",
    "Osmanlı Devleti'nin kurucusu kimdir?",
    "Kurtuluş Savaşı hangi yıllar arasında olmuştur?",
    "Berlin Duvarı hangi yıl yıkılmıştır?",
    "Sümerler hangi bölgede kurulmuştur?",

    # 🔬 Science
    "DNA nedir?",
    "Evrende en bol bulunan element nedir?",
    "Suyun kaynama noktası kaç derecedir?",
    "Bir atomun çekirdeğinde ne bulunur?",
    "Hangi organ oksijeni vücuda taşır?",
    "Işığın hızı saniyede kaç kilometredir?",
    "Bir yılı yaklaşık kaç gün olarak kabul ederiz?",
    "Einstein'ın ünlü formülü nedir?",

    # 🌍 Geography
    "Türkiye'nin en uzun nehri nedir?",
    "Afrika kıtasının en büyük ülkesi hangisidir?",
    "Dünya'nın en yüksek dağı nedir?",
    "Amazon Nehri hangi kıtadadır?",
    "İstanbul iki kıta arasında köprü görevi görür. Bu kıtalar hangileridir?",
    "Dünya'nın en büyük çölü hangisidir?",
    "Türkiye'nin başkenti neresidir?",
    "Kutup bölgelerinde yaşayan ayı türü hangisidir?",

    # 📖 General knowledge
    "Nobel Barış Ödülü hangi şehirde verilir?",
    "FIFA Dünya Kupası kaç yılda bir düzenlenir?",
    "Mona Lisa tablosunun ressamı kimdir?",
    "En çok konuşulan dil hangisidir?",
    "Japonya'nın başkenti neresidir?",
    "Hangi ülke en fazla yüzölçümüne sahiptir?",
    "Olimpiyatların sembolünde kaç halka vardır?",
    "Venedik hangi ülkeye aittir?",

    # 🧬 Health
    "İnsan vücudunda kaç kemik vardır?",
    "Normal insan kalp atış hızı dakikada kaçtır?",
    "İnsan vücudunda en büyük organ nedir?",
    "Diyabet hangi organın bozulması sonucu oluşur?",
    "Kırmızı kan hücrelerinin diğer adı nedir?",

    # 📚 Literature
    "Cahit Sıtkı Tarancı hangi eseriyle ünlüdür?",
    "Suç ve Ceza romanı kimin eseridir?",
    "William Shakespeare'in en ünlü trajedilerinden biri nedir?",
    "Küçük Prens kitabının yazarı kimdir?",
    "İnce Memed romanını kim yazmıştır?",

    # ➗ Math
    "7 ile 8 çarpımı kaç eder?",
    "Pi sayısının yaklaşık değeri nedir?",
    "Bir üçgenin iç açılarının toplamı kaç derecedir?",
    "2'nin küpü kaçtır?",
    "1000 sayısının yarısı kaçtır?",

    # 💻 Technology
    "İlk programlanabilir bilgisayarın adı nedir?",
    "İnternetin temel protokolü nedir?",
    "OpenAI tarafından geliştirilen ünlü dil modeli nedir?"
]

In [36]:
# Open a .txt file for writing
with open("model_test_sonuclari.txt", "w", encoding="utf-8") as f:
    for idx, soru in enumerate(sorular, 1):
        input_text = f"Soru: {soru} Kısa ve net bir cevap ver.\nCevap:"
        
        # Tokenize
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

        # Generate answer
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

        # Decode
        output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Write to file
        f.write("============================\n")
        f.write(f"Soru {idx}: {soru}\n")
        f.write("Modelin cevabı:\n")
        f.write(output_text)
        f.write("\n============================\n\n")

print("✅ All answers saved to 'model_test_sonuclari.txt'!")

✅ All answers saved to 'model_test_sonuclari.txt'!


### Generate answers next to the reference answers
Answers were then graded manually (`results/model_answers/`).

In [37]:
import csv

# Our test questions and REFERENCE answers
sorular_ve_cevaplar = [
    ("İstanbul'un fethi hangi yılda gerçekleşmiştir?", "1453"),
    ("Atatürk hangi yılda doğmuştur?", "1881"),
    ("Magna Carta hangi yıl imzalanmıştır?", "1215"),
    ("Fransız Devrimi kaç yılında olmuştur?", "1789"),
    ("Osmanlı Devleti'nin kurucusu kimdir?", "Osman Gazi"),
    ("Kurtuluş Savaşı hangi yıllar arasında olmuştur?", "1919-1923"),
    ("Berlin Duvarı hangi yıl yıkılmıştır?", "1989"),
    ("Sümerler hangi bölgede kurulmuştur?", "Mezopotamya"),
    ("DNA nedir?", "Genetik bilgiyi taşıyan moleküldür"),
    ("Evrende en bol bulunan element nedir?", "Hidrojen"),
    ("Suyun kaynama noktası kaç derecedir?", "100°C"),
    ("Bir atomun çekirdeğinde ne bulunur?", "Proton ve nötron"),
    ("Hangi organ oksijeni vücuda taşır?", "Akciğer"),
    ("Işığın hızı saniyede kaç kilometredir?", "Yaklaşık 300,000 km/s"),
    ("Bir yılı yaklaşık kaç gün olarak kabul ederiz?", "365 gün"),
    ("Einstein'ın ünlü formülü nedir?", "E=mc^2"),
    ("Türkiye'nin en uzun nehri nedir?", "Kızılırmak"),
    ("Afrika kıtasının en büyük ülkesi hangisidir?", "Cezayir"),
    ("Dünya'nın en yüksek dağı nedir?", "Everest"),
    ("Amazon Nehri hangi kıtadadır?", "Güney Amerika"),
    ("İstanbul iki kıta arasında köprü görevi görür. Bu kıtalar hangileridir?", "Asya ve Avrupa"),
    ("Dünya'nın en büyük çölü hangisidir?", "Antarktika"),
    ("Türkiye'nin başkenti neresidir?", "Ankara"),
    ("Kutup bölgelerinde yaşayan ayı türü hangisidir?", "Kutup ayısı"),
    ("Nobel Barış Ödülü hangi şehirde verilir?", "Oslo"),
    ("FIFA Dünya Kupası kaç yılda bir düzenlenir?", "4"),
    ("Mona Lisa tablosunun ressamı kimdir?", "Leonardo da Vinci"),
    ("En çok konuşulan dil hangisidir?", "İngilizce"),
    ("Japonya'nın başkenti neresidir?", "Tokyo"),
    ("Hangi ülke en fazla yüzölçümüne sahiptir?", "Rusya"),
    ("Olimpiyatların sembolünde kaç halka vardır?", "5"),
    ("Venedik hangi ülkeye aittir?", "İtalya"),
    ("İnsan vücudunda kaç kemik vardır?", "206"),
    ("Normal insan kalp atış hızı dakikada kaçtır?", "60-100"),
    ("İnsan vücudunda en büyük organ nedir?", "Deri"),
    ("Diyabet hangi organın bozulması sonucu oluşur?", "Pankreas"),
    ("Kırmızı kan hücrelerinin diğer adı nedir?", "Eritrosit"),
    ("Cahit Sıtkı Tarancı hangi eseriyle ünlüdür?", "Otuz Beş Yaş"),
    ("Suç ve Ceza romanı kimin eseridir?", "Dostoyevski"),
    ("William Shakespeare'in en ünlü trajedilerinden biri nedir?", "Hamlet"),
    ("Küçük Prens kitabının yazarı kimdir?", "Antoine de Saint-Exupéry"),
    ("İnce Memed romanını kim yazmıştır?", "Yaşar Kemal"),
    ("7 ile 8 çarpımı kaç eder?", "56"),
    ("Pi sayısının yaklaşık değeri nedir?", "3.14"),
    ("Bir üçgenin iç açılarının toplamı kaç derecedir?", "180"),
    ("2'nin küpü kaçtır?", "8"),
    ("1000 sayısının yarısı kaçtır?", "500"),
    ("İlk programlanabilir bilgisayarın adı nedir?", "ENIAC"),
    ("İnternetin temel protokolü nedir?", "TCP/IP"),
    ("OpenAI tarafından geliştirilen ünlü dil modeli nedir?", "GPT")
]

# Open the CSV file
with open("model_test_sonuclari.csv", mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["Soru", "Modelin Cevabı", "Gerçek Cevap"])  # Write the header row

    for soru, gercek_cevap in sorular_ve_cevaplar:
        input_text = f"Soru: {soru} Kısa ve net bir cevap ver.\nCevap:"
        
        # Tokenize
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

        # Generate answer from the model
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

        # Decode
        output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Trim the input to keep only the answer
        cevap = output_text.split("Cevap:")[-1].strip()

        # Write to CSV
        writer.writerow([soru, cevap, gercek_cevap])

print("✅ Tüm cevaplar 'model_test_sonuclari.csv' dosyasına Soru | Model Cevabı | Gerçek Cevap olarak kaydedildi!")

✅ Tüm cevaplar 'model_test_sonuclari.csv' dosyasına Soru | Model Cevabı | Gerçek Cevap olarak kaydedildi!


## Run 2 · GPT-2 large on GPT-4o answers

In [38]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer
from peft import LoraConfig
from datasets import load_dataset
import torch

# 1. Model and tokenizer
model_id = "ytu-ce-cosmos/turkish-gpt2-large"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    load_in_8bit=False,
    load_in_4bit=False
)

# 2. Dataset
dataset = load_dataset("json", data_files="v1_dataset_gpt4o.jsonl", split="train")
dataset = dataset.map(lambda example: {"completion": example["response"]})

# 3. LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["c_attn", "c_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 4. Training arguments
training_args = TrainingArguments(
    output_dir="./gpt2-large-finetuned-gpt4o",
    per_device_train_batch_size=1,  # Small batch because the model is large
    gradient_accumulation_steps=8,  # For more stable training
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    num_train_epochs=3,
    bf16=True,
)

# 5. Formatting function
def formatting_prompts(example):
    return f"Soru: {example['prompt']}\nCevap: {example['completion']}"

# 6. Trainer setup
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    formatting_func=formatting_prompts,
    args=training_args,
)

# 7. Start training
trainer.train()

Step,Training Loss
10,2.707400
20,2.559600
30,2.466400
40,2.486900
50,2.475000
60,2.358200
70,2.447900
80,2.456300
90,2.435400
100,2.457500


TrainOutput(global_step=5208, training_loss=2.1857643273935157, metrics={'train_runtime': 4908.9778, 'train_samples_per_second': 8.489, 'train_steps_per_second': 1.061, 'total_flos': 2.54971281411072e+16, 'train_loss': 2.1857643273935157})

In [39]:
# Save the large model and tokenizer
trainer.model.save_pretrained("./gpt2-large-finetuned-gpt4o-final")
tokenizer.save_pretrained("./gpt2-large-finetuned-gpt4o-final")

('./gpt2-large-finetuned-gpt4o-final/tokenizer_config.json',
 './gpt2-large-finetuned-gpt4o-final/special_tokens_map.json',
 './gpt2-large-finetuned-gpt4o-final/vocab.json',
 './gpt2-large-finetuned-gpt4o-final/merges.txt',
 './gpt2-large-finetuned-gpt4o-final/added_tokens.json',
 './gpt2-large-finetuned-gpt4o-final/tokenizer.json')

In [40]:
# Load the large model and tokenizer
model_path = "./gpt2-large-finetuned-gpt4o-final"
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_path)

# CSV'ye yaz
with open("model_test_sonuclari_gpt4o_large.csv", mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["Soru", "Modelin Cevabı", "Gerçek Cevap"])

    for soru, gercek_cevap in sorular_ve_cevaplar:
        input_text = f"Soru: {soru} Kısa ve net bir cevap ver.\nCevap:"
        
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

        output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        cevap = output_text.split("Cevap:")[-1].strip()

        writer.writerow([soru, cevap, gercek_cevap])
print("✅ Tüm cevaplar 'model_test_sonuclari_gpt4o_large.csv' dosyasına Soru | Model Cevabı | Gerçek Cevap olarak kaydedildi!")

✅ Tüm cevaplar 'model_test_sonuclari_gpt4o_large.csv' dosyasına Soru | Model Cevabı | Gerçek Cevap olarak kaydedildi!


## Run 3 · GPT-2 medium on DeepSeek answers

In [41]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer
from peft import LoraConfig
from datasets import load_dataset
import torch

# 1. Model and tokenizer
model_id = "ytu-ce-cosmos/turkish-gpt2-medium"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    load_in_8bit=False,
    load_in_4bit=False
)

# 2. Dataset
dataset = load_dataset("json", data_files="v2_dataset_deepseek.jsonl", split="train")
dataset = dataset.map(lambda example: {"completion": example["response"]})

# 3. LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["c_attn", "c_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 4. Training arguments
training_args = TrainingArguments(
    output_dir="./gpt2-medium-finetuned-deepseek",
    per_device_train_batch_size=2,  # Small batch because the model is large
    gradient_accumulation_steps=4,  # For more stable training
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    num_train_epochs=3,
    bf16=True,
)

# 5. Formatting function
def formatting_prompts(example):
    return f"Soru: {example['prompt']}\nCevap: {example['completion']}"

# 6. Trainer setup
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    formatting_func=formatting_prompts,
    args=training_args,
)

# 7. Start training
trainer.train()

Step,Training Loss
10,3.017500
20,2.916600
30,2.840200
40,2.869500
50,2.788800
60,2.659100
70,2.650200
80,2.665000
90,2.671700
100,2.675200


TrainOutput(global_step=5208, training_loss=2.4056074498435867, metrics={'train_runtime': 1924.8511, 'train_samples_per_second': 21.65, 'train_steps_per_second': 2.706, 'total_flos': 2.03958155114496e+16, 'train_loss': 2.4056074498435867})

In [42]:
trainer.model.save_pretrained("./gpt2-medium-finetuned-deepseek-final")
tokenizer.save_pretrained("./gpt2-medium-finetuned-deepseek-final")

('./gpt2-medium-finetuned-deepseek-final/tokenizer_config.json',
 './gpt2-medium-finetuned-deepseek-final/special_tokens_map.json',
 './gpt2-medium-finetuned-deepseek-final/vocab.json',
 './gpt2-medium-finetuned-deepseek-final/merges.txt',
 './gpt2-medium-finetuned-deepseek-final/added_tokens.json',
 './gpt2-medium-finetuned-deepseek-final/tokenizer.json')

In [44]:
import csv
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load the DeepSeek model and tokenizer
model_path = "./gpt2-medium-finetuned-deepseek-final"
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Save DeepSeek results to CSV
with open("model_test_sonuclari_medium_deepseek.csv", mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["Soru", "Modelin Cevabı", "Gerçek Cevap"])

    for soru, gercek_cevap in sorular_ve_cevaplar:
        input_text = f"Soru: {soru} Kısa ve net bir cevap ver.\nCevap:"
        
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

        output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        cevap = output_text.split("Cevap:")[-1].strip()

        writer.writerow([soru, cevap, gercek_cevap])

print("✅ Deepseek model cevapları 'model_test_sonuclari_medium_deepseek.csv' dosyasına kaydedildi!")

✅ Deepseek model cevapları 'model_test_sonuclari_medium_deepseek.csv' dosyasına kaydedildi!


## Run 4 · GPT-2 large on DeepSeek answers

In [45]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer
from peft import LoraConfig
from datasets import load_dataset
import torch

# 1. Model and tokenizer
model_id = "ytu-ce-cosmos/turkish-gpt2-large"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    load_in_8bit=False,
    load_in_4bit=False
)

# 2. Dataset
dataset = load_dataset("json", data_files="v2_dataset_deepseek.jsonl", split="train")
dataset = dataset.map(lambda example: {"completion": example["response"]})

# 3. LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["c_attn", "c_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 4. Training arguments
training_args = TrainingArguments(
    output_dir="./gpt2-large-finetuned-deepseek",
    per_device_train_batch_size=1,  # Small batch because the model is large
    gradient_accumulation_steps=8,  # For more stable training
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    num_train_epochs=3,
    bf16=True,
)

# 5. Formatting function
def formatting_prompts(example):
    return f"Soru: {example['prompt']}\nCevap: {example['completion']}"

# 6. Trainer setup
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    formatting_func=formatting_prompts,
    args=training_args,
)

# 7. Start training
trainer.train()

Step,Training Loss
10,2.936900
20,2.795100
30,2.719600
40,2.753700
50,2.668700
60,2.558500
70,2.537100
80,2.545400
90,2.584200
100,2.553800


TrainOutput(global_step=5208, training_loss=2.279950479208599, metrics={'train_runtime': 5092.8441, 'train_samples_per_second': 8.183, 'train_steps_per_second': 1.023, 'total_flos': 3.166816191822336e+16, 'train_loss': 2.279950479208599})

In [47]:
trainer.model.save_pretrained("./gpt2-large-finetuned-deepseek-final")
tokenizer.save_pretrained("./gpt2-large-finetuned-deepseek-final")

('./gpt2-large-finetuned-deepseek-final/tokenizer_config.json',
 './gpt2-large-finetuned-deepseek-final/special_tokens_map.json',
 './gpt2-large-finetuned-deepseek-final/vocab.json',
 './gpt2-large-finetuned-deepseek-final/merges.txt',
 './gpt2-large-finetuned-deepseek-final/added_tokens.json',
 './gpt2-large-finetuned-deepseek-final/tokenizer.json')

In [48]:
import csv
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load the DeepSeek model and tokenizer
model_path = "./gpt2-large-finetuned-deepseek-final"
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Save DeepSeek results to CSV
with open("model_test_sonuclari_large_deepseek.csv", mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["Soru", "Modelin Cevabı", "Gerçek Cevap"])

    for soru, gercek_cevap in sorular_ve_cevaplar:
        input_text = f"Soru: {soru} Kısa ve net bir cevap ver.\nCevap:"
        
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

        output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        cevap = output_text.split("Cevap:")[-1].strip()

        writer.writerow([soru, cevap, gercek_cevap])

print("✅ Deepseek model cevapları 'model_test_sonuclari_large_deepseek.csv' dosyasına kaydedildi!")

✅ Deepseek model cevapları 'model_test_sonuclari_large_deepseek.csv' dosyasına kaydedildi!
